# P4a - fine-tuning smoke test (Qwen3.5-4B, bf16 LoRA)

Adapted from the official Unsloth `Qwen3_5_(4B)_Vision.ipynb` (fetched 2026-08-22).
Context: `ai-collab/plans/2026-08-15_track-vlm-parser.md`, stage P4.

**This is a plumbing test, not a training run.** 120 samples, 50 steps. It passes when

1. the install completes and an Ampere+ GPU is present (L4 = capability 8.9),
2. loss moves,
3. the adapter saves,
4. **a per-step time is measured** - which is what sizes the real run and its cost.

Quality is deliberately *not* judged here: 50 steps over 116 samples cannot teach
anything. That is P4c.

The runtime is billed while connected, idle included. Use
`Runtime -> Disconnect and delete runtime` when finished.

## 1. Install (several minutes)

Commands copied verbatim from the official Unsloth `Qwen3_5_(4B)_Vision.ipynb`
(fetched 2026-08-22). Do **not** "modernise" the pins: the stack is built against
torch 2.8.0 while Colab currently defaults to 2.11.0+cu128, and the CUDA extensions
only build against the pinned one. Expect several minutes and possibly a restart
prompt.

Two deliberate differences from upstream:

- **`%%capture` removed.** Upstream hides the install output to keep the notebook
  tidy. That is the wrong trade for a smoke test whose entire job is to surface
  problems early: with it, a failed install is swallowed and resurfaces later as a
  confusing `import unsloth` error. The output is long; that is the point.
- The explanation lives in this markdown cell rather than above the code, because a
  cell magic has to be the very first line of its cell.

In [2]:
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 134.9 MB/s eta 0:00:0000:0100:01
Using Python 3.12.13 environment at: /usr
Resolved 4 packages in 105ms                                         
Prepared 1 package in 62ms                                               
Uninstalled 1 package in 3ms
Installed 1 package in 9ms                                  
 - trl==0.24.0
 + trl==0.22.2
Using Python 3.12.13 environment at: /usr
Resolved 28 packages in 143ms                                        
Prepared 2 packages in 473ms                                             
Uninstalled 1 package in 76ms
Installed 2 packages in 48ms                                
 - transformers==5.5.0
 + transformers==5.2.0
 + typer-slim==0.24.0
Using Python 3.12.13 environment at: /usr
Resolved 55 packages in 217ms                                        
Prepared 4 packages in 14.59s                                            
Installed 4 packages in 276ms                               
 + causal-co

## 2. Refuse the wrong runtime

In [3]:
# Refuse the wrong runtime here rather than discover it an hour into a run.
import torch

capability = torch.cuda.get_device_capability(0)
print("device       ", torch.cuda.get_device_name(0))
print("capability   ", capability)
print("VRAM GiB     ", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("bf16 NATIVE  ", torch.cuda.is_bf16_supported(including_emulation=False))

# `is_bf16_supported()` defaults to including_emulation=True and answers True on a
# Turing T4, which has no bf16 hardware at all. Always ask for the native answer.
assert capability >= (8, 0), (
    f"Got capability {capability}; this needs Ampere or newer (L4 is 8.9). "
    "Runtime -> Disconnect and delete runtime, then reconnect having picked L4."
)

device        NVIDIA L4
capability    (8, 9)
VRAM GiB      22.03
bf16 NATIVE   True


## 3. Verify the upload, then copy it off Drive

In [4]:
import hashlib
import json
from pathlib import Path

from google.colab import drive

DRIVE_DIR = Path("/content/drive/MyDrive/colab_finetune")
ARCHIVE = "zip_vl_6x6_smoke120_20260822.tar"
DATASET = "smoke_6x6"
EXPECTED_SHA256 = "4424ecc88907173d57b6a7569f68bb259d8a4f4f86da3d1ef523cf0cd10df266"

drive.mount("/content/drive")

source = DRIVE_DIR / ARCHIVE
digest = hashlib.sha256(source.read_bytes()).hexdigest()
print("sha256   ", digest)
print("expected ", EXPECTED_SHA256)
assert digest == EXPECTED_SHA256, "Upload is corrupt, or this is the wrong archive."

# Copy onto the VM's own disk before reading. Pulling thousands of small files through
# the Drive FUSE mount throttles the dataloader badly.
!cp "{source}" /content/
!tar -xf /content/{ARCHIVE} -C /content/
!ls /content/{DATASET} && ls /content/{DATASET}/images | wc -l

Mounted at /content/drive
sha256    4424ecc88907173d57b6a7569f68bb259d8a4f4f86da3d1ef523cf0cd10df266
expected  4424ecc88907173d57b6a7569f68bb259d8a4f4f86da3d1ef523cf0cd10df266
images	manifest.json  metadata.jsonl
120


## 4. Build conversations

In [5]:
from PIL import Image

DATA_DIR = Path(f"/content/{DATASET}")

# Verbatim from src/core/vl_models/prompt_variants.FINETUNE_INSTRUCTION.
# Train and infer with the SAME string -- a checkpoint queried with the baseline
# few-shot prompt is being asked a question it never saw.
INSTRUCTION = 'Read this Zip puzzle screenshot and reply with ONLY a JSON object.\n"layout" is a 2D array of two-character strings: "  " for an empty cell, "xx" for a blocked cell, and a zero-padded number such as "01" for a waypoint.\n"walls" is a list of {"cell1": [row, col], "cell2": [row, col]} objects, one per thick black bar drawn on a grid line between two neighbouring cells. Report every wall you can see and do not invent any.'

records = [
    json.loads(line)
    for line in (DATA_DIR / "metadata.jsonl").read_text("utf-8").splitlines()
]
print(len(records), "records")


def to_conversation(record):
    image = Image.open(DATA_DIR / record["file_name"]).convert("RGB")
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": INSTRUCTION},
                    {"type": "image", "image": image},
                ],
            },
            {"role": "assistant", "content": [{"type": "text", "text": record["label"]}]},
        ]
    }


# Held out so the after-training check is not just reciting a training sample.
holdout, train_records = records[:4], records[4:]
train_dataset = [to_conversation(record) for record in train_records]
print("train", len(train_dataset), " holdout", len(holdout))
print(train_records[0]["label"][:200], "...")

120 records
train 116  holdout 4
{
  "layout": [
    ["  ", "  ", "04", "  ", "05", "06"],
    ["  ", "  ", "  ", "03", "02", "  "],
    ["01", "  ", "  ", "  ", "  ", "  "],
    ["09", "  ", "  ", "  ", "  ", "  "],
    ["  ", "  ", ...


## 5. Model and LoRA

In [6]:
from unsloth import FastVisionModel

# load_in_4bit=False -> 16-bit LoRA. Unsloth advises against QLoRA for Qwen3.5
# (larger than usual quantisation error), which is exactly why this route needs
# native bf16, and therefore a paid L4 rather than the free T4.
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",
    load_in_4bit = False,
    use_gradient_checkpointing = "unsloth",
)

/usr/local/lib/python3.12/dist-packages/unsloth/_gpu_init.py:98: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: /usr/local/lib/python3.12/dist-packages/torchaudio/lib/_torchaudio.abi3.so: undefined symbol: torch_library_impl
  disable_torchaudio_if_cuda_mismatched()


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.19: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [7]:
model = FastVisionModel.get_peft_model(
    model,
    # The failure this project is trying to fix is purely visual: numbers and layout
    # are already read correctly (cell accuracy 0.96 untuned) and what is missed are
    # the black bars pressed onto the grid lines (wall F1 0.31). Freezing the vision
    # tower to save memory would very likely learn nothing that matters here.
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 6. Behaviour before training

Same held-out image as the after-check, so the comparison is honest rather than
an impression.

In [8]:
from transformers import TextStreamer


def answer(record, max_new_tokens=400):
    """Greedy decode for one record, printed as it streams."""
    FastVisionModel.for_inference(model)
    image = Image.open(DATA_DIR / record["file_name"]).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": INSTRUCTION},
    ]}]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(image, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True)
    _ = model.generate(**inputs, streamer=streamer, max_new_tokens=max_new_tokens,
                       use_cache=True, do_sample=False)


print("=== BEFORE training, held-out image ===")
print("--- ground truth ---")
print(holdout[0]["label"])
print("--- model ---")
answer(holdout[0])

=== BEFORE training, held-out image ===
--- ground truth ---
{
  "layout": [
    ["  ", "  ", "  ", "  ", "  ", "  "],
    ["  ", "01", "  ", "04", "  ", "  "],
    ["08", "09", "  ", "03", "  ", "  "],
    ["  ", "  ", "  ", "  ", "  ", "05"],
    ["  ", "  ", "02", "  ", "  ", "06"],
    ["  ", "  ", "  ", "07", "  ", "  "]
  ],
  "walls": [
    {"cell1": [0, 2], "cell2": [0, 3]},
    {"cell1": [1, 1], "cell2": [1, 2]},
    {"cell1": [1, 2], "cell2": [1, 3]},
    {"cell1": [1, 4], "cell2": [1, 5]},
    {"cell1": [2, 1], "cell2": [2, 2]},
    {"cell1": [2, 1], "cell2": [3, 1]},
    {"cell1": [3, 2], "cell2": [3, 3]},
    {"cell1": [3, 2], "cell2": [4, 2]},
    {"cell1": [4, 0], "cell2": [4, 1]},
    {"cell1": [4, 1], "cell2": [5, 1]},
    {"cell1": [4, 2], "cell2": [5, 2]},
    {"cell1": [4, 3], "cell2": [5, 3]}
  ]
}
--- model ---
The user wants me to extract information from a Zip puzzle image.

**1. Analyze the Grid:**
- It's a 6x6 grid.
- I need to identify the content of each cel

## 7. Train

In [9]:
from trl import SFTConfig, SFTTrainer
from unsloth.trainer import UnslothVisionDataCollator

FastVisionModel.for_training(model)

MAX_STEPS = 50   # A plumbing test, not a training run.
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 4

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer),
    train_dataset = train_dataset,
    args = SFTConfig(
        per_device_train_batch_size = PER_DEVICE_BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps = 5,
        max_steps = MAX_STEPS,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",

        # Required for vision fine-tuning; the collator prepares each batch itself.
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        # Longest label over the full 8,000-sample set is 769 characters (~250 tokens),
        # so image tokens dominate this budget rather than the answer.
        max_length = 2048,
    ),
)
print(round(torch.cuda.max_memory_reserved() / 1024**3, 3), "GB reserved before training")

Unsloth: Model does not have a default image size - using 512
9.004 GB reserved before training


In [10]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 116 | Num Epochs = 4 | Total steps = 50
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 38,756,352 of 4,578,021,888 (0.85% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,1.010879
2,1.034796
3,1.072235
4,0.957571
5,0.729524
6,0.580967
7,0.510432
8,0.451365
9,0.324340
10,0.222448


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-50/tokenizer_config.json.


## 8. Measure and extrapolate

In [11]:
# This cell is the actual deliverable of P4a: the numbers that decide whether the real
# run stays on an L4, and roughly what it will cost.
runtime = trainer_stats.metrics["train_runtime"]
seconds_per_step = runtime / MAX_STEPS
peak = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
total = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 3)

effective_batch = PER_DEVICE_BATCH * GRAD_ACCUM
FULL_DATASET = 8000
for epochs in (1, 2, 3):
    steps = FULL_DATASET * epochs / effective_batch
    print(f"{epochs} epoch(s): {steps:6.0f} steps -> {steps * seconds_per_step / 3600:5.2f} h")

print()
print(f"measured   {seconds_per_step:.2f} s/step over {MAX_STEPS} steps ({runtime:.0f}s)")
print(f"peak VRAM  {peak} / {total} GiB  ({peak / total * 100:.1f}%)")
print()
print("Multiply the hours by the L4 compute-unit rate in the Colab runtime picker for")
print("the cost. If peak VRAM is well under total, per_device_train_batch_size can go")
print("up and the hours come down.")

1 epoch(s):   1000 steps ->  2.09 h
2 epoch(s):   2000 steps ->  4.19 h
3 epoch(s):   3000 steps ->  6.28 h

measured   7.54 s/step over 50 steps (377s)
peak VRAM  16.568 / 22.034 GiB  (75.2%)

Multiply the hours by the L4 compute-unit rate in the Colab runtime picker for
the cost. If peak VRAM is well under total, per_device_train_batch_size can go
up and the hours come down.


## 9. Behaviour after training

In [12]:
print("=== AFTER training, same held-out image ===")
print("--- ground truth ---")
print(holdout[0]["label"])
print("--- model ---")
answer(holdout[0])

=== AFTER training, same held-out image ===
--- ground truth ---
{
  "layout": [
    ["  ", "  ", "  ", "  ", "  ", "  "],
    ["  ", "01", "  ", "04", "  ", "  "],
    ["08", "09", "  ", "03", "  ", "  "],
    ["  ", "  ", "  ", "  ", "  ", "05"],
    ["  ", "  ", "02", "  ", "  ", "06"],
    ["  ", "  ", "  ", "07", "  ", "  "]
  ],
  "walls": [
    {"cell1": [0, 2], "cell2": [0, 3]},
    {"cell1": [1, 1], "cell2": [1, 2]},
    {"cell1": [1, 2], "cell2": [1, 3]},
    {"cell1": [1, 4], "cell2": [1, 5]},
    {"cell1": [2, 1], "cell2": [2, 2]},
    {"cell1": [2, 1], "cell2": [3, 1]},
    {"cell1": [3, 2], "cell2": [3, 3]},
    {"cell1": [3, 2], "cell2": [4, 2]},
    {"cell1": [4, 0], "cell2": [4, 1]},
    {"cell1": [4, 1], "cell2": [5, 1]},
    {"cell1": [4, 2], "cell2": [5, 2]},
    {"cell1": [4, 3], "cell2": [5, 3]}
  ]
}
--- model ---
The user wants me to extract a Zip puzzle screenshot into a JSON format.
"layout" is a 2D array of two-character strings: "  " for an empty cell, "xx" 

## 10. Save the adapter

In [13]:
# The LoRA adapter only. Merged 16-bit and GGUF exports are P4d, and GGUF has a known
# vision-export defect (unsloth#3899) that needs checking before it is relied on.
OUT = "/content/drive/MyDrive/colab_finetune/qwen35_4b_zip_smoke_lora"
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
!du -sh "{OUT}"

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/colab_finetune/qwen35_4b_zip_smoke_lora/tokenizer_config.json.


168M	/content/drive/MyDrive/colab_finetune/qwen35_4b_zip_smoke_lora


In [14]:
import torch

vis, lang, other = [], [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    (vis if ".visual." in name else lang if ".language_model." in name else other).append((name, p))

print(f"trainable tensors: visual={len(vis)} language={len(lang)} other={len(other)}")
print(f"trainable params : visual={sum(p.numel() for _,p in vis):,} "
      f"language={sum(p.numel() for _,p in lang):,} other={sum(p.numel() for _,p in other):,}")

for label, group in (("visual", vis), ("language", lang)):
    bs = [(n, p) for n, p in group if "lora_B" in n]
    if not bs:
        print(f"{label}: NO lora_B"); continue
    nonzero = sum(1 for _, p in bs if p.abs().max().item() > 0)
    print(f"{label:9s}: {nonzero}/{len(bs)} lora_B non-zero, "
          f"max|B|={max(p.abs().max().item() for _,p in bs):.3e}")

# --- 渲染比對：訓練用的字串 vs 推論用的字串 ---
conv = train_dataset[0]["messages"]
train_text = tokenizer.apply_chat_template(conv, tokenize=False)
infer_default = tokenizer.apply_chat_template([conv[0]], add_generation_prompt=True, tokenize=False)
infer_nothink = tokenizer.apply_chat_template([conv[0]], add_generation_prompt=True,
                                              tokenize=False, enable_thinking=False)
print("\n--- TRAIN ends with ---\n", repr(train_text[-200:]))
print("\n--- INFER (default) ends with ---\n", repr(infer_default[-200:]))
print("\n--- INFER (enable_thinking=False) ends with ---\n", repr(infer_nothink[-200:]))


trainable tensors: visual=192 language=496 other=0
trainable params : visual=6,291,456 language=32,464,896 other=0
visual   : 96/96 lora_B non-zero, max|B|=1.137e-01
language : 248/248 lora_B non-zero, max|B|=5.866e-02

--- TRAIN ends with ---
 '2, 4], "cell2": [2, 5]},\n    {"cell1": [3, 2], "cell2": [4, 2]},\n    {"cell1": [3, 4], "cell2": [3, 5]},\n    {"cell1": [4, 0], "cell2": [4, 1]},\n    {"cell1": [5, 1], "cell2": [5, 2]}\n  ]\n}<|im_end|>\n'

--- INFER (default) ends with ---
 ' black bar drawn on a grid line between two neighbouring cells. Report every wall you can see and do not invent any.<|vision_start|><|image_pad|><|vision_end|><|im_end|>\n<|im_start|>assistant\n<think>\n'

--- INFER (enable_thinking=False) ends with ---
 'drawn on a grid line between two neighbouring cells. Report every wall you can see and do not invent any.<|vision_start|><|image_pad|><|vision_end|><|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'


## Before P4c on the full set

**Do not just swap in the 8,000-sample archive and re-run cell 4.** It decodes every
image into a PIL object held in a Python list; at roughly 1.2 MB per decoded image
that is about 10 GB of system RAM against the ~12.7 GB a standard Colab VM has. At 120
samples it is irrelevant; at 8,000 it will die.

The full run needs the dataset to stay lazy - an `imagefolder` `datasets.Dataset`
with a transform instead of a materialised list. **Verify that on a short run before
starting a multi-hour one**, not during it.

## 11. E0 - does the rendering fix work?

The P4a run exposed **two** mismatches between how a sample is rendered for training and
how the prompt is rendered for inference:

| | training | inference (as first written) |
|---|---|---|
| thinking | no `<think>` at all - the answer follows `assistant\n` directly | `<think>\n`, opened and never closed |
| content order | `[text, image]` | `[image, text]` |

Either one alone is enough to put the model somewhere it was never trained. Together they
explain the P4a result exactly: the content was *correct* (every wall named matched ground
truth) but it arrived as prose inside a thinking block instead of as JSON.

`build_inference_prompt` fixes both by construction: it renders a conversation through the
**same** template call training uses and cuts at the answer, so the prefix is identical by
definition rather than by careful copying. Note that `enable_thinking=False` would *not*
have been enough - it emits `<think>\n\n</think>\n\n`, which training never saw either.

The 4 held-out samples never appeared in training, but they are still synthetic. This
measures "did it learn our renderer", **not** "can it read a real screenshot".

In [ ]:
import json
import re


def build_inference_prompt(tokenizer, instruction):
    """Render through the SAME template call training uses, then cut at the answer.

    Guarantees the prompt prefix matches training byte for byte, instead of relying on
    the inference-side arguments being kept in sync by hand.
    """
    sentinel = "@@ANSWER@@"
    conv = [
        {"role": "user", "content": [{"type": "text", "text": instruction},
                                     {"type": "image"}]},
        {"role": "assistant", "content": [{"type": "text", "text": sentinel}]},
    ]
    return tokenizer.apply_chat_template(conv, tokenize=False).split(sentinel)[0]


def predict(record, max_new_tokens=600):
    FastVisionModel.for_inference(model)
    image = Image.open(DATA_DIR / record["file_name"]).convert("RGB")
    prompt = build_inference_prompt(tokenizer, INSTRUCTION)
    inputs = tokenizer(image, prompt, add_special_tokens=False,
                       return_tensors="pt").to("cuda")
    generated = model.generate(**inputs, max_new_tokens=max_new_tokens,
                               use_cache=True, do_sample=False)
    return tokenizer.decode(generated[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True)


def wall_set(payload):
    return {tuple(sorted((tuple(w["cell1"]), tuple(w["cell2"]))))
            for w in payload["walls"]}


def score(prediction, truth):
    """Layout exactness plus wall precision/recall/F1 - the metric that matters."""
    predicted_walls, true_walls = wall_set(prediction), wall_set(truth)
    hits = len(predicted_walls & true_walls)
    precision = hits / len(predicted_walls) if predicted_walls else (0.0 if true_walls else 1.0)
    recall = hits / len(true_walls) if true_walls else 1.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "layout_exact": prediction.get("layout") == truth["layout"],
        "wall_hits": hits,
        "walls_true": len(true_walls),
        "walls_pred": len(predicted_walls),
        "wall_f1": f1,
    }


print("prompt now ends with:", repr(build_inference_prompt(tokenizer, INSTRUCTION)[-64:]))
print()

parsed = 0
for record in holdout:
    raw = predict(record)
    truth = json.loads(record["label"])
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        print(f"{record['file_name']}: NO JSON   raw={raw[:110]!r}")
        continue
    try:
        prediction = json.loads(match.group(0))
    except json.JSONDecodeError as error:
        print(f"{record['file_name']}: BAD JSON  {error}")
        continue
    parsed += 1
    result = score(prediction, truth)
    print(f"{record['file_name']}: layout={'OK' if result['layout_exact'] else 'X '} "
          f"walls {result['wall_hits']}/{result['walls_true']} "
          f"(predicted {result['walls_pred']})  F1={result['wall_f1']:.3f}")

print(f"\nJSON parsed: {parsed}/{len(holdout)}")

## 12. E1 - how much is image resolution costing?

P4a measured **7.54 s/step** at an effective batch of 8, i.e. ~0.94 s per sample. That is
slow for a 4B model on an L4, and the most likely reason is resolution: the dataset renders
at `cell_size` 72-132, giving images from roughly 500 to 950 px, and a vision encoder bills
by patch count.

Rather than time training runs, this counts **tokens per sample** at several resolutions.
Token count is what drives the compute, it is instant to measure, and it does not train the
adapter further and contaminate it.

If halving resolution roughly halves the token count, it multiplies through every run that
follows. Walls are thick black bars, so they may well survive the downscale - but that is a
question for accuracy, measured separately, not for this cell.

In [ ]:
from PIL import Image

probe_record = holdout[0]
original = Image.open(DATA_DIR / probe_record["file_name"]).convert("RGB")
prompt = build_inference_prompt(tokenizer, INSTRUCTION)

# Same s/step figure P4a measured, used to turn token counts into a time estimate.
BASELINE_SECONDS_PER_STEP = 7.54

print(f"source image: {original.size}")
print(f"{'longest side':>13}{'size':>13}{'tokens':>9}{'vs base':>9}{'est s/step':>12}")

baseline_tokens = None
for longest_side in (None, 768, 640, 512, 448, 384):
    if longest_side is None:
        image = original
    else:
        scale = longest_side / max(original.size)
        image = original.resize(
            (max(1, round(original.width * scale)), max(1, round(original.height * scale))),
            Image.Resampling.LANCZOS,
        )
    tokens = tokenizer(image, prompt, add_special_tokens=False,
                       return_tensors="pt")["input_ids"].shape[1]
    if baseline_tokens is None:
        baseline_tokens = tokens
    ratio = tokens / baseline_tokens
    label = "original" if longest_side is None else str(longest_side)
    print(f"{label:>13}{str(image.size):>13}{tokens:>9}{ratio:>8.2f}x"
          f"{BASELINE_SECONDS_PER_STEP * ratio:>11.2f}s")

print()
print("Token count is a proxy for compute, not a promise: attention is superlinear, so a")
print("real run may gain more than this suggests. Accuracy at each size is a separate")
print("question - measure it before choosing one.")

## When this notebook is finished

`Runtime -> Disconnect and delete runtime`. The runtime bills at the same rate whether it
is training or idle (measured 2026-08-22: L4 = 1.54 compute units/hour), so an idle session
left overnight costs about as much as the entire P4 stage.

Reusable across a reconnect: nothing. Reusable **within** this runtime: the installed
packages and the downloaded weights, which is the expensive part. To train a second variant
cleanly, call `FastVisionModel.from_pretrained` again for a fresh base model - do **not**
keep training the adapter above, which has already seen the no-CoD data.